# Урок 1.4. Первый запрос к LLM: работаем с API как инженеры

**Модуль 1 «Введение в LLM» · Курс «LLM для Python-разработчиков»**

В уроке 1.2 мы ходили в API «голым» httpx — чтобы увидеть протокол своими глазами. В уроке 1.3 выяснили, что OpenAI-совместимый API стал отраслевым стандартом. Пора собрать это вместе и написать запрос так, как он выглядит в настоящем продакшн-коде: со стандартным SDK, таймаутами, обработкой ошибок, ретраями, стримингом и учётом токенов.

## 🎯 Цели урока

После этого урока вы сможете:

- делать запросы через официальный **`openai` SDK** — один и тот же код для OpenAI, OpenRouter, vLLM и Ollama;
- читать **объект ответа**: `choices`, `finish_reason`, `usage`;
- различать **транзиентные и постоянные ошибки** и понимать, какие из них ретраить;
- писать **retry с экспоненциальным backoff и джиттером** — и знать, когда хватит встроенного в SDK;
- включать **стриминг** и объяснять, зачем он нужен;
- делать **параллельные запросы** через `AsyncOpenAI` + `asyncio.gather`;
- собрать всё в **переиспользуемый мини-клиент** — он поедет с нами в следующие уроки.

## План

1. От httpx к SDK
2. Настройка клиента и первый запрос
3. Таксономия ошибок
4. Ретраи с backoff
5. Стриминг
6. Учёт токенов и денег
7. Асинхронность
8. Сборка мини-клиента

⏱ **Время:** ~75–90 минут.

> 💡 Ollama мы установим только в уроке 1.5, поэтому «живые» ячейки здесь вежливо пропускаются, а рядом в тексте показан реальный вывод модели. Зато вся инженерная обвязка — ошибки, ретраи, backoff, валидация — работает по-настоящему прямо сейчас. **После урока 1.5 вернитесь и перезапустите ноутбук целиком.**

## 1. От httpx к SDK

В уроке 1.2 мы собирали HTTP-запросы руками. Это было правильно — как разок поговорить с Postgres через голый socket, чтобы понять протокол. Но в проде так никто не делает: для баз данных берут драйвер, для LLM — SDK.

Официальная библиотека `openai` даёт:

- **единый интерфейс** для всех OpenAI-совместимых провайдеров (урок 1.3): смена провайдера = смена `base_url`;
- **типизированные объекты** вместо словарей: `response.choices[0].message.content`, и IDE подсказывает поля;
- **встроенные ретраи и таймауты** — меньше самодельной обвязки;
- **стриминг и async** из коробки.

А знание «сырого» протокола останется с вами навсегда: когда SDK бросит непонятную ошибку, вы будете знать, что у него под капотом обычный POST с JSON.

In [ ]:
# Зависимости урока: openai SDK и httpx (для проверки доступности Ollama).
# %pip install -q openai httpx

In [ ]:
import httpx

OLLAMA_URL = "http://localhost:11434"

try:
    httpx.get(OLLAMA_URL, timeout=2)
    OLLAMA_AVAILABLE = True
    print("Ollama запущена — все живые ячейки урока будут работать.")
except (httpx.ConnectError, httpx.TimeoutException):
    OLLAMA_AVAILABLE = False
    print("Ollama недоступна. Это нормально: мы установим её в уроке 1.5,")
    print("а пока живые ячейки будут вежливо пропускаться.")
    print("После урока 1.5 вернитесь и перезапустите ноутбук целиком.")

## 2. Настраиваем клиент

Клиент конфигурируется четырьмя вещами — и все они должны жить в конфигурации, а не в коде (как DSN базы данных):

| параметр | для Ollama | зачем |
|---|---|---|
| `base_url` | `http://localhost:11434/v1` | OpenAI-совместимый эндпоинт Ollama |
| `api_key` | `"ollama"` (любая непустая строка) | локально ключ не проверяется, но SDK требует поле |
| `timeout` | 60+ секунд | помните prefill на CPU из урока 1.3? Дефолтные таймауты рассчитаны на облако |
| `max_retries` | пока `0` | ретраи сначала напишем сами — чтобы понимать, что делает SDK |

🔐 Из урока 1.2: реальные ключи **никогда** не попадают в код и git — только переменные окружения. Здесь `"ollama"` — публичная заглушка, ей утекать некуда.

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",   # заглушка: локально не проверяется, но поле обязательно
    timeout=60.0,       # щедрый лимит: CPU-инференс нетороплив (урок 1.3)
    max_retries=0,      # встроенные ретраи выключим - напишем свои и сравним
)
MODEL = "qwen2.5:3b"

print("Клиент создан:", client.base_url)

Обратите внимание: создание клиента — **ленивое**, никакого похода в сеть не случилось. Соединение откроется при первом запросе.

## 3. Первый запрос

Метод — `client.chat.completions.create()`. Параметры вам уже знакомы по уроку 1.2, только имена «стандартные»: `max_tokens` вместо `num_predict`. В ответ приходит не словарь, а типизированный объект:

```text
response
├── id, model, created            # метаданные
├── choices[0]
│   ├── message
│   │   ├── role      = "assistant"
│   │   └── content   = текст ответа
│   └── finish_reason = "stop" | "length" | ...
└── usage
    ├── prompt_tokens             # сколько токенов съел промпт
    ├── completion_tokens         # сколько сгенерировано
    └── total_tokens
```

`choices` — список, потому что API умеет генерировать несколько вариантов ответа за один запрос (параметр `n`); почти всегда вариант один, и вы пишете `choices[0]`.

In [ ]:
if OLLAMA_AVAILABLE:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Ты — лаконичный ассистент. Отвечай одним-двумя предложениями."},
            {"role": "user", "content": "Что такое HTTP простыми словами?"},
        ],
        temperature=0.7,
        max_tokens=120,
    )
    print(response.choices[0].message.content)
else:
    print("[пропущено: Ollama недоступна — вернитесь после урока 1.5]")

> **Пример вывода** (qwen2.5:3b, реальный прогон):
>
> «HTTP — это язык, на котором браузер разговаривает с сервером: клиент отправляет запрос ("дай мне эту страницу"), а сервер возвращает ответ с кодом статуса и содержимым.»

Кроме текста, в ответе едут служебные поля — и в продакшене вы будете смотреть на них не реже, чем на сам текст. `finish_reason == "length"` — тот самый сигнал «ответ обрезан по `max_tokens`» (в родном API Ollama он назывался `done_reason`), а `usage` — основа учёта расходов:

In [ ]:
if OLLAMA_AVAILABLE:
    print("finish_reason:", response.choices[0].finish_reason)
    print("model:        ", response.model)
    u = response.usage
    print(f"usage: {u.prompt_tokens} промпт + {u.completion_tokens} ответ = {u.total_tokens} токенов")
else:
    print("[пропущено: Ollama недоступна — вернитесь после урока 1.5]")
    print("Ожидаемый вывод:")
    print("  finish_reason: stop")
    print("  model:         qwen2.5:3b")
    print("  usage: 37 промпт + 52 ответ = 89 токенов")

## 4. Что может пойти не так: таксономия ошибок

LLM-провайдер — это внешний сервис, и относиться к нему нужно как к любому внешнему API: **он будет падать, тормозить и троттлить вас**, вопрос только когда. Жизненный цикл запроса и где что ломается:

```text
create() ──> [соединение] ──> [отправка] ──> [ожидание] ──> [HTTP-статус] ──> [парсинг]
                  |                              |                |
        APIConnectionError            APITimeoutError    4xx/5xx -> APIStatusError
       (сервер лежит, сеть, DNS)     (не уложились         и его подклассы
                                      в timeout)
```

Главная классификация — **транзиентные** ошибки (пройдут сами, можно повторить) против **постоянных** (повторяй хоть сто раз — не поможет, чини код):

| исключение SDK | HTTP | причина | ретраить? |
|---|---|---|---|
| `APIConnectionError` | — | сервер недоступен, сеть, DNS | ✅ да |
| `APITimeoutError` | — | не уложились в `timeout` | ✅ да (и пересмотреть timeout) |
| `RateLimitError` | 429 | слишком много запросов | ✅ да, обязательно с backoff |
| `InternalServerError` | 5xx | провайдеру плохо | ✅ да |
| `AuthenticationError` | 401 | неверный ключ | ❌ нет — чинить конфиг |
| `BadRequestError` | 400 | кривой запрос: имя модели, переполненный контекст | ❌ нет — чинить код |

Ретраить `BadRequestError` — классическая ошибка: вы будете долбить провайдера заведомо сломанным запросом и маскировать собственный баг.

Проверим обработку ошибок **по-настоящему** — Ollama для этого не нужна: сходим на порт, где никто не слушает, как будто наш LLM-сервис упал:

In [ ]:
from openai import APIConnectionError

broken_client = OpenAI(
    base_url="http://localhost:9999/v1",  # здесь никто не слушает - "упавший" сервис
    api_key="ollama",
    timeout=3.0,
    max_retries=0,
)

try:
    broken_client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "Привет!"}],
    )
except APIConnectionError as e:
    print("Поймали:", type(e).__name__)
    print("ОС мгновенно ответила 'connection refused' - на порту 9999 пусто.")
    print("Ровно так же выглядит упавший прод: ловим, логируем, ретраим.")

## 5. Ретраи с экспоненциальным backoff

Транзиентную ошибку хочется повторить. Но повторять **сразу же** — плохая идея: если провайдер захлебнулся (429), мгновенные ретраи от тысяч клиентов его добивают. Стандартное решение из мира распределённых систем — **экспоненциальный backoff с джиттером**:

- **экспонента**: пауза удваивается с каждой попыткой (`base × 2^attempt`), давая сервису время прийти в себя;
- **потолок** (cap): чтобы пауза не выросла до бесконечности;
- **джиттер** (jitter): случайный разброс пауз, чтобы клиенты, упавшие одновременно, не вернулись тоже одновременно (эффект «толпы» — thundering herd).

In [ ]:
import random

random.seed(42)  # только ради воспроизводимого вывода; в проде seed не нужен

def backoff_delay(attempt: int, base: float = 0.5, cap: float = 8.0) -> float:
    """Пауза перед попыткой attempt (счёт с нуля): экспонента + джиттер ±50%."""
    delay = min(cap, base * 2 ** attempt)
    return delay * random.uniform(0.5, 1.5)

for attempt in range(5):
    print(f"после попытки {attempt + 1}: ждём ~{backoff_delay(attempt):.2f} с")

Теперь обернём вызов в retry-цикл и проверим его в деле — опять на «упавшем» сервисе, так что демонстрация честная:

In [ ]:
import time

from openai import APITimeoutError, InternalServerError, RateLimitError

# APITimeoutError - подкласс APIConnectionError; указан явно для наглядности
RETRIABLE = (APIConnectionError, APITimeoutError, RateLimitError, InternalServerError)

def with_retries(fn, max_attempts: int = 3):
    """Вызывает fn, повторяя ТОЛЬКО транзиентные ошибки с backoff."""
    for attempt in range(max_attempts):
        try:
            return fn()
        except RETRIABLE as e:
            if attempt == max_attempts - 1:
                raise  # попытки кончились - пробрасываем ошибку наверх
            delay = backoff_delay(attempt, base=0.3, cap=2.0)
            print(f"  попытка {attempt + 1}/{max_attempts}: {type(e).__name__}, ждём {delay:.1f} с")
            time.sleep(delay)

try:
    with_retries(lambda: broken_client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "Привет!"}],
    ))
except APIConnectionError:
    print("Все попытки исчерпаны. Дальше - fallback: честная ошибка пользователю,")
    print("деградация до кэшированного ответа или очередь на потом.")

> 💡 **В реальном проекте начните со встроенного:** `OpenAI(max_retries=2)` — SDK сам ретраит ошибки соединения, 408, 429 и 5xx с экспоненциальным backoff. Свой цикл нужен, когда хочется логирования, метрик или особых правил (например, «на 429 ждать столько, сколько сказал заголовок `Retry-After`»). Для сложных случаев есть библиотека `tenacity`. Но теперь вы знаете, что именно делает за вас `max_retries` — а значит, сможете это отлаживать.

## 6. Стриминг: токены по мере генерации

Из урока 1.3 вы помните: генерация — это токен за токеном, и ответ в 300 токенов занимает секунды. Без стриминга пользователь смотрит на спиннер всё это время; со стримингом первые слова появляются почти сразу — как в интерфейсах ChatGPT и других чатов. Для чат-UX это разница между «работает» и «зависло».

Технически: передаём `stream=True`, и вместо одного ответа получаем **итератор чанков** (под капотом — server-sent events). В каждом чанке `delta.content` — очередной кусочек текста:

In [ ]:
if OLLAMA_AVAILABLE:
    stream = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "Сочини четверостишие про программиста и кота."}],
        max_tokens=120,
        stream=True,
    )
    parts = []
    for chunk in stream:
        if not chunk.choices:      # некоторые провайдеры шлют служебные чанки без choices
            continue
        delta = chunk.choices[0].delta.content or ""
        parts.append(delta)
        print(delta, end="", flush=True)   # текст появляется по мере генерации
    print(f"\n\nПолучено чанков: примерно по одному на токен, символов: {len(''.join(parts))}")
else:
    print("[пропущено: Ollama недоступна — вернитесь после урока 1.5]")

> **Пример вывода** (появляется постепенно, слово за словом):
>
> «Ночью тихо кот мурлычет, / Программист свой код печатает, / Баг находит — кот мяучит, / Вместе релиз приближают.»

Когда стриминг **не** нужен: батч-обработка без пользователя у экрана и случаи, где ответ парсится целиком (JSON из урока 1.2 — пока не пришла последняя скобка, парсить нечего). Правило простое: стримим для людей, ждём целиком для машин.

## 7. Учёт токенов и денег

`usage` приходит с каждым ответом — и его стоит **логировать всегда**, даже на бесплатной локальной модели: сегодня это привычка, завтра — строка в счёте. Зная цены из урока 1.3, превратим `usage` в деньги:

In [ ]:
PRICES_USD_PER_1M = {   # из урока 1.3: (input, output) за 1 млн токенов
    "gpt-4o-mini":   (0.15, 0.60),
    "claude-sonnet": (3.00, 15.00),
    "qwen2.5:3b":    (0.00, 0.00),   # локальная модель: за токены не платим
}

def request_cost_usd(model: str, prompt_tokens: int, completion_tokens: int) -> float:
    price_in, price_out = PRICES_USD_PER_1M.get(model, (0.0, 0.0))
    return (prompt_tokens / 1e6 * price_in
            + completion_tokens / 1e6 * price_out)

# usage из живого ответа - или сохранённый пример, если Ollama недоступна
if OLLAMA_AVAILABLE:
    usage = {"prompt_tokens": response.usage.prompt_tokens,
             "completion_tokens": response.usage.completion_tokens}
else:
    usage = {"prompt_tokens": 37, "completion_tokens": 52}  # из реального прогона

print(f"Запрос: {usage['prompt_tokens']} in + {usage['completion_tokens']} out\n")
for model in PRICES_USD_PER_1M:
    cost = request_cost_usd(model, **usage)
    print(f"{model:<14} ${cost:.6f}")

## 8. Асинхронность: параллельные запросы

Вы бэкенд-разработчик, поэтому знаете: блокировать обработчик FastAPI синхронным вызовом на 10 секунд — преступление. У SDK есть зеркальный асинхронный клиент `AsyncOpenAI` — те же методы, только с `await`.

Классическая задача «обработать пачку текстов» решается через `asyncio.gather` — запросы уходят параллельно:

> ⚠️ **Честное предупреждение про локальный CPU:** Ollama по умолчанию выполняет запросы последовательно (одна модель — одна очередь), так что чудо-ускорения на ноутбуке не ждите. Но код ниже — ровно тот же, что даст кратный выигрыш с облачным API или vLLM, у которых на стороне сервера настоящий параллелизм.

Техническая деталь для ноутбуков: в Jupyter event loop уже занят самим ядром, поэтому `asyncio.run()` в ячейке падает. Обойдём это честным способом — уведём корутину в отдельный поток со своим циклом (в FastAPI такой костыль не нужен: там вы просто пишете `await`):

In [ ]:
import asyncio
import threading

from openai import AsyncOpenAI

def run_async(coro):
    """Запускает корутину и в обычном скрипте, и в Jupyter (где loop уже занят)."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)          # обычный python-скрипт: просто запускаем
    box = {}                              # Jupyter: свой цикл в отдельном потоке
    thread = threading.Thread(target=lambda: box.update(value=asyncio.run(coro)))
    thread.start()
    thread.join()
    return box["value"]

REVIEWS = [
    "Отличный сервис, всё быстро!",
    "Ужасно: заказ потеряли, поддержка молчит.",
    "Нормально, но доставка могла быть быстрее.",
]

async def classify(aclient: AsyncOpenAI, text: str) -> str:
    resp = await aclient.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Определи тональность отзыва. Ответь одним словом: позитив, негатив или нейтрально."},
            {"role": "user", "content": text},
        ],
        temperature=0.0,
        max_tokens=10,
    )
    return resp.choices[0].message.content.strip()

async def classify_all(texts: list[str]) -> list[str]:
    async with AsyncOpenAI(base_url="http://localhost:11434/v1",
                           api_key="ollama", timeout=120.0) as aclient:
        return await asyncio.gather(*(classify(aclient, t) for t in texts))

if OLLAMA_AVAILABLE:
    start = time.perf_counter()
    labels = run_async(classify_all(REVIEWS))
    for review, label in zip(REVIEWS, labels):
        print(f"[{label}] {review}")
    print(f"\nОбработано за {time.perf_counter() - start:.1f} с")
else:
    print("[пропущено: Ollama недоступна — вернитесь после урока 1.5]")

> **Пример вывода:**
>
> ```text
> [позитив] Отличный сервис, всё быстро!
> [негатив] Ужасно: заказ потеряли, поддержка молчит.
> [нейтрально] Нормально, но доставка могла быть быстрее.
>
> Обработано за 4.8 с
> ```

Важный нюанс продакшена: `gather` на **сотню** запросов разом — это способ мгновенно словить `RateLimitError` от облака (или положить свой локальный сервер). Параллелизм ограничивают семафором:

```python
sem = asyncio.Semaphore(5)          # не больше 5 запросов одновременно

async def one(text):
    async with sem:
        return await classify(aclient, text)
```

Этот паттерн — в упражнении 3.

## 9. Собираем всё: мини-клиент для проекта

Соберём выводы урока в один класс: конфигурация из переменных окружения (как DSN базы), встроенные ретраи SDK, накопительный учёт токенов. Упрощённая версия этого клиента будет нашим рабочим инструментом в следующих уроках:

In [ ]:
import os

class LLMClient:
    """Мини-обёртка над любым OpenAI-совместимым API."""

    def __init__(self):
        self.client = OpenAI(
            base_url=os.getenv("LLM_BASE_URL", "http://localhost:11434/v1"),
            api_key=os.getenv("LLM_API_KEY", "ollama"),
            timeout=float(os.getenv("LLM_TIMEOUT_S", "60")),
            max_retries=2,               # транзиентные ошибки ретраит сам SDK
        )
        self.model = os.getenv("LLM_MODEL", "qwen2.5:3b")
        self.total_prompt_tokens = 0
        self.total_completion_tokens = 0

    def ask(self, user_text: str, *,
            system: str = "Ты — полезный ассистент.",
            temperature: float = 0.7,
            max_tokens: int = 500) -> str:
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user_text},
            ],
            temperature=temperature,
            max_tokens=max_tokens,
        )
        if response.usage:               # копим статистику по всем запросам
            self.total_prompt_tokens += response.usage.prompt_tokens
            self.total_completion_tokens += response.usage.completion_tokens
        if response.choices[0].finish_reason == "length":
            print("[внимание: ответ обрезан по max_tokens]")
        return response.choices[0].message.content

llm = LLMClient()
print(f"Клиент готов: {llm.client.base_url} / модель {llm.model}")
print("Смена провайдера = переменные окружения LLM_BASE_URL / LLM_API_KEY / LLM_MODEL")

In [ ]:
if OLLAMA_AVAILABLE:
    print(llm.ask("Назови три причины использовать очередь сообщений в бэкенде.",
                  max_tokens=2000))
    print(f"\nНакоплено токенов: {llm.total_prompt_tokens} промпт / "
          f"{llm.total_completion_tokens} ответ")
else:
    print("[пропущено: Ollama недоступна — вернитесь после урока 1.5]")

## ⚠️ Частые ошибки

1. **Ключи и URL захардкожены в коде.** Ключ, попавший в git, считается скомпрометированным навсегда. Конфигурация — в переменных окружения, в коде — только `os.getenv`.
2. **Нет таймаута или он «облачный».** Дефолтные таймауты рассчитаны на быстрые GPU провайдера; локальный CPU-инференс в них не укладывается, и вы получаете ложные «падения» на здоровом сервисе.
3. **Ретраить всё подряд.** Повторять `BadRequestError` или `AuthenticationError` бессмысленно — это ваши баги, а ретраи их маскируют. Повторяют только транзиентное: соединение, таймаут, 429, 5xx.
4. **Ретраи без backoff и джиттера.** Мгновенные повторы добивают перегруженный сервис и превращают лёгкий инцидент в тяжёлый. Экспонента + потолок + джиттер.
5. **Игнорировать `usage` и `finish_reason`.** Без `usage` вы не узнаете, какая фича сжигает бюджет; без `finish_reason` будете отдавать пользователям обрезанные на полуслове ответы.
6. **Стриминг не там.** В чате без стриминга интерфейс «висит»; в пайплайне с JSON-парсингом стриминг бесполезен — ответ всё равно нужен целиком.
7. **`asyncio.gather` без ограничения параллелизма.** Сто одновременных запросов — это `RateLimitError` от облака или захлебнувшийся локальный сервер. Семафор — обязательная часть паттерна.

## 🏋️ Упражнения

### Упражнение 1 — стриминг с секундомером ⏱

Измерьте два времени для стримингового запроса: **TTFT** (time to first token — от отправки до первого непустого чанка) и полное время генерации. Выведите оба и сам ответ. Если Ollama ещё недоступна — напишите код сейчас, проверите после урока 1.5.

### Упражнение 2 — `ask_json` на SDK 📦

Перенесите паттерн из урока 1.2 на SDK: функция `ask_json(llm, user_text, schema_model)`, которая просит модель ответить JSON-объектом (`response_format={"type": "json_object"}`), валидирует ответ Pydantic-моделью и при `ValidationError` делает одну повторную попытку, скормив модели текст ошибки. Валидацию схемы можно проверить прямо сейчас — на «записанном» ответе модели.

### Упражнение 3 — устойчивый батч 🚦

Обработайте список отзывов асинхронно, но правильно: параллелизм ограничен `asyncio.Semaphore(2)`, ошибки отдельных запросов не роняют весь батч (`asyncio.gather(..., return_exceptions=True)`). Выведите результат по каждому тексту и итог: сколько успешно, сколько с ошибкой.

## 📝 Мини-квиз

**1. Зачем передавать `api_key="ollama"`, если Ollama ключи не проверяет?**

<details><summary>Ответ</summary>

SDK требует непустой ключ — это обязательное поле клиента. Локальному серверу оно безразлично, поэтому подставляют любую строку-заглушку. Утечь ей некуда: секретом она не является.

</details>

**2. Какие ошибки имеет смысл ретраить, а какие — нет?**

<details><summary>Ответ</summary>

Ретраят **транзиентные**: `APIConnectionError`, `APITimeoutError`, `RateLimitError` (429), `InternalServerError` (5xx). Не ретраят **постоянные**: 400 (кривой запрос) и 401 (ключ) — их повтор лишь маскирует баг и нагружает провайдера.

</details>

**3. Зачем в backoff добавляют джиттер?**

<details><summary>Ответ</summary>

Чтобы клиенты, отвалившиеся одновременно, не вернулись тоже одновременно (thundering herd). Случайный разброс пауз размазывает повторную нагрузку во времени.

</details>

**4. Что означает `finish_reason == "length"` и что с этим делать?**

<details><summary>Ответ</summary>

Ответ обрезан по `max_tokens` (в родном API Ollama — `done_reason`). Варианты: увеличить лимит, попросить модель отвечать короче или явно обработать обрезку. Молча отдавать пользователю обрубленный текст — худший вариант.

</details>

**5. Почему `asyncio.gather` почти не ускоряет запросы к локальной Ollama на CPU, но ускоряет запросы к облаку?**

<details><summary>Ответ</summary>

Локально одна модель обрабатывает запросы по очереди — параллелить нечем. У облака (и vLLM) на стороне сервера кластер и батчинг — параллельные запросы реально идут одновременно. Код при этом одинаковый, поэтому писать его стоит сразу асинхронно.

</details>

## ✅ Решения упражнений

Сначала попробуйте сами!

### Решение 1 — стриминг с секундомером

In [ ]:
if OLLAMA_AVAILABLE:
    start = time.perf_counter()
    ttft = None
    parts = []
    stream = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "Объясни в трёх предложениях, что такое REST."}],
        max_tokens=150,
        stream=True,
    )
    for chunk in stream:
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta.content or ""
        if delta and ttft is None:
            ttft = time.perf_counter() - start   # первый непустой чанк
        parts.append(delta)
    total = time.perf_counter() - start
    text = "".join(parts)
    print(text)
    print(f"\nTTFT: {ttft:.2f} с | полное время: {total:.2f} с | символов: {len(text)}")
else:
    print("[пропущено: Ollama недоступна — вернитесь после урока 1.5]")

TTFT — это как раз «сеть + prefill» из урока 1.3, а разница между полным временем и TTFT — чистая генерация. Померив оба числа, вы знаете, что оптимизировать: долгий TTFT лечится укорачиванием промпта, долгая генерация — уменьшением `max_tokens`.

### Решение 2 — `ask_json` на SDK

In [ ]:
from pydantic import BaseModel, Field, ValidationError

class Sentiment(BaseModel):
    label: str = Field(pattern="^(позитив|негатив|нейтрально)$")
    confidence: float = Field(ge=0, le=1)

def ask_json(llm: LLMClient, user_text: str, schema_model, *, max_attempts: int = 2):
    """JSON-режим + Pydantic-валидация + повтор с текстом ошибки (урок 1.2, теперь на SDK)."""
    messages = [
        {"role": "system", "content": (
            "Отвечай строго JSON-объектом с полями label "
            "(позитив|негатив|нейтрально) и confidence (число от 0 до 1).")},
        {"role": "user", "content": user_text},
    ]
    for _ in range(max_attempts):
        response = llm.client.chat.completions.create(
            model=llm.model,
            messages=messages,
            temperature=0.0,
            max_tokens=100,
            response_format={"type": "json_object"},   # JSON-режим в OpenAI-совместимом API
        )
        raw = response.choices[0].message.content
        try:
            return schema_model.model_validate_json(raw)
        except ValidationError as e:
            # скармливаем модели её же ошибку - пусть чинит сама
            messages.append({"role": "assistant", "content": raw})
            messages.append({"role": "user", "content": f"Исправь JSON. Ошибки валидации: {e}"})
    raise ValueError("Модель так и не вернула валидный JSON")

# Схему можно проверить прямо сейчас, без модели - на "записанном" ответе:
saved = '{"label": "негатив", "confidence": 0.93}'
print("Схема работает:", Sentiment.model_validate_json(saved))

if OLLAMA_AVAILABLE:
    result = ask_json(llm, "Отзыв: 'Заказ пришёл раздавленным, верните деньги! ЧЕРТИ!'", Sentiment)
    print("Живой ответ модели:", result)
else:
    print("[живой вызов пропущен: Ollama недоступна — вернитесь после урока 1.5]")

### Решение 3 — устойчивый батч

In [ ]:
async def classify_batch(texts: list[str], max_concurrency: int = 2):
    sem = asyncio.Semaphore(max_concurrency)
    async with AsyncOpenAI(base_url="http://localhost:11434/v1",
                           api_key="ollama", timeout=120.0) as aclient:

        async def one(text: str) -> str:
            async with sem:                    # не больше max_concurrency одновременно
                return await classify(aclient, text)

        # return_exceptions=True: упавший запрос кладёт в результат исключение,
        # а не роняет весь gather
        return await asyncio.gather(*(one(t) for t in texts), return_exceptions=True)

if OLLAMA_AVAILABLE:
    texts = REVIEWS * 2                        # 6 текстов, параллелизм - 2
    results = run_async(classify_batch(texts))
    ok = sum(1 for r in results if not isinstance(r, Exception))
    for text, r in zip(texts, results):
        mark = f"ошибка: {type(r).__name__}" if isinstance(r, Exception) else r
        print(f"[{mark}] {text}")
    print(f"\nУспешно: {ok} из {len(texts)}")
else:
    print("[пропущено: Ollama недоступна — вернитесь после урока 1.5]")

Тройка `Semaphore + gather + return_exceptions` — рабочий скелет любой батч-обработки через LLM: она не бомбит провайдера, не умирает от одного неудачного запроса и даёт полную картину по итогам. В модуле про RAG мы будем этим скелетом индексировать документы.

## 🎓 Итоги

- **SDK `openai`** — стандартный клиент для любого OpenAI-совместимого API: конфигурация (`base_url`, ключ, модель, `timeout`) живёт в окружении, код не меняется при смене провайдера.
- Ответ — типизированный объект: текст в `choices[0].message.content`, диагностика в `finish_reason`, деньги в `usage`.
- Ошибки делятся на **транзиентные** (ретраить: соединение, таймаут, 429, 5xx) и **постоянные** (чинить: 400, 401). Ретраи — с экспоненциальным backoff, потолком и джиттером; в простых случаях хватает `max_retries` самого SDK.
- **Стриминг** — для людей у экрана (TTFT решает), целый ответ — для машин и JSON.
- **`AsyncOpenAI` + `gather` + `Semaphore`** — паттерн параллельной обработки; локально на CPU выигрыш скромный, но код тот же, что для облака.
- Всё собрано в `LLMClient` — он поедет с нами дальше по курсу.

**Дальше — урок 1.5 «Установка и запуск LLM локально с помощью Ollama»:** наконец-то поставим Ollama, скачаем модель и вернёмся сюда запустить все живые ячейки.

## 📚 Что почитать

- [openai-python](https://github.com/openai/openai-python) — README библиотеки: ретраи, таймауты, стриминг, async
- [API reference: chat completions](https://platform.openai.com/docs/api-reference/chat) — все параметры запроса и поля ответа
- [Ollama: OpenAI compatibility](https://ollama.com/blog/openai-compatibility) — что именно из API поддержано
- [tenacity](https://tenacity.readthedocs.io/) — библиотека ретраев, когда своего цикла станет мало
- [AWS Architecture Blog: Exponential Backoff and Jitter](https://aws.amazon.com/blogs/architecture/exponential-backoff-and-jitter/) — классическая статья о том, почему джиттер обязателен